[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/GSTT-CSC/summer-school-tutorials/blob/day2/day2_qms_regs/QMS_runthrough_day2.ipynb)

# CSC Summer School Day 2 — Quality Management Systems & Regulations

## Clinical context: HandOsteo

**HandOsteo** is an as-yet unreleased Software as a Medical Device (SaMD) that automatically screens patients for osteoporosis and osteopenia from routine AP (anteroposterior) hand X-rays. It is currently in development by the GSTT-CSC Team

Osteoporosis is a skeletal disease characterised by reduced bone mineral density (BMD), affecting approximately 3.5 million people in the UK and causing around 500,000 fragility fractures per year. Early detection is critical — patients with known osteoporosis can be treated with bisphosphonates or lifestyle interventions that substantially reduce fracture risk.

A validated, low-cost screening proxy for BMD is the **Metacarpal Cortical Percentage (MCP)**: the ratio of cortical bone thickness to total bone width across the shaft of the second metacarpal, measured on a plain AP hand X-ray.

```
        |<------- A (total width) ------->|
        |<- B/2 ->|            |<- B/2 ->|
        ┌─────────┬────────────┬──────────┐
        │ cortex  │  medullary │  cortex  │
        └─────────┴────────────┴──────────┘
        
        MCP = ((A − B) / A) × 100
```

A MCP below ~75 % is associated with osteopenia; below ~65 % with osteoporosis.

### The HandOsteo pipeline

We have created a very basic copy of what the HandOsteo pipeline might look like:

| Step | Class | What it does |
|------|-------|--------------|
| 1 | `DicomLoader` | Load and validate the modality of the DICOM (must be CR or DX) |
| 2 | `ViewClassifier` | Confirm the view is AP or PA; reject oblique views |
| 3 | `MCPMeasurer` | Segment the second metacarpal and compute A, B, and MCP |
| 4 | `ReportGenerator` | Create a structured report and route it to PACS |

Hand X-rays are ideal for this application because they are: (a) already acquired routinely for arthritis and injury assessment, meaning no additional radiation dose is needed for screening; (b) inexpensive compared with DEXA scanning; and (c) read rapidly by a software tool, removing the bottleneck of radiologist review for a screening programme.

---

## What you will do today

You will act as both the developer and the quality engineer for HandOsteo and produce a lightweight **Quality Management System (QMS)**, based on the [GSTT CSC QMS template](https://github.com/GSTT-CSC/QMS-Template):

| # | Stage | What you will produce |
|---|-------|-----------------------|
| 1 | **Project Initiation** | Device description, classification, and role assignments |
| 2 | **Requirements Gathering** | System Requirements Specification (SRS) from the care process |
| 3 | **Hazard Log** | Risk assessment linked back to requirements |
| 4 | **Design Specification** | System Design Specification (SDS) items that implement each SRS |
| 5 | **Verification** | Automated unit tests that confirm the code meets the SDS |
| 6 | **Validation** | Manual test cases that confirm the system meets the SRS |

All artefacts are stored as YAML and rendered into QMS documents via a `make` pipeline — mirroring how the GSTT CSC QMS Template works.


In [ ]:
%pip install -q rdm pandas pytest pyyaml pydicom ipytest pillow dicom2nifti nibabel svgwrite ipython

In [ ]:
import os, sys

# Fetch course data and helpers if not already present
if not os.path.isdir('day2_data') or not os.path.exists('care_process_diagram.py'):
    !git clone --depth=1 -b day2 https://github.com/GSTT-CSC/summer-school-tutorials.git _repo
    !cp -rn _repo/day2_qms_regs/day2_data .
    !cp _repo/day2_qms_regs/care_process_diagram.py .
    !rm -rf _repo

# Ensure local helpers are importable (needed for some Jupyter environments)
if '' not in sys.path:
    sys.path.insert(0, '')

print('Data ready:', os.path.isdir('day2_data'))
print('Diagram helper ready:', os.path.exists('care_process_diagram.py'))

from zipfile import ZipFile
import yaml
import pandas as pd
from pathlib import Path
import pydicom
import dicom2nifti
import nibabel
import matplotlib.pyplot as plt
import pytest
import ipytest

class MyDumper(yaml.Dumper):

    def increase_indent(self, flow=False, indentless=False):
        return super(MyDumper, self).increase_indent(flow, False)

> ### 💡 Stuck? Load the example solutions
>
> If you'd like to see a worked example for HandOsteo, run the cell below.
> It will overwrite the `data/` files with completed versions — you can then read through them
> and use them as a reference for the rest of the tutorial.
>
> ⚠️ This will replace anything you have already written in the data files.

In [ ]:
# 💡 Load the worked HandOsteo solutions into data/
# Set LOAD_SOLUTIONS = True and run this cell only if you want to replace your work with the example answers.

LOAD_SOLUTIONS = False  # <-- change to True to load solutions

if LOAD_SOLUTIONS:
    import shutil
    from pathlib import Path
    for src in Path('day2_data/data_solutions').iterdir():
        shutil.copy(src, f'day2_data/data/{src.name}')
    print('Solutions loaded — refresh the file browser (↺) to see them.')
else:
    print('LOAD_SOLUTIONS is False — nothing changed. Set it to True above if you want the example answers.')


### HandOsteo Care Process Diagram

The diagram below shows a proposed care pathway for HandOsteo — automated osteoporosis/osteopenia screening from AP hand X-rays.

In [ ]:
from IPython.display import SVG, display
from src.care_process_diagram import build_diagram

display(SVG(data=build_diagram()))

> ### ✏️ How to edit files in Colab
>
> Several steps below ask you to edit a `.yml` file. In Colab:
>
> 1. Click the **📁 Files** icon in the left sidebar.
> 2. Open the **`day2_data/data/`** folder.
> 3. **Double-click** a file (e.g. `device.yml`) to open it in the editor pane on the right.
> 4. Make your edits, then press **Ctrl/Cmd-S** to save.
>
> ⚠️ Colab storage is temporary — if your runtime disconnects you'll lose your edits and the data, and need to re-run the setup cells above.

# 1. Project Initiation

Open `day2_data/data/device.yml` and fill in:
- A **product name** and **version number**
- The **device classification** (class I, IIa, IIb, or III under UK MDR 2002)

Then open `day2_data/data/people.yml` and assign roles (Developer, Quality Engineer, Clinical Lead).

In [ ]:
# display the contents of the device yaml file.
with open("day2_data/data/device.yml", 'r') as stream:
    data_loaded = yaml.safe_load(stream)

print(yaml.dump(data_loaded, Dumper=MyDumper, default_flow_style=False))

# 2. Requirements Gathering

Open `day2_data/data/requirements.yml` and add a new system requirement based on the care process diagram above. The first requirement has been completed as an example.

Each requirement needs:
- A short **title** and **description**
- A **unique identifier** (e.g. `SRS_002`)
- A **requirement type**

### Rules for writing good requirements

- **Clear, concise, and unambiguous** — one interpretation only
- **Measurable** — must be verifiable by an automated unit test
- Focus on **what** the software must do, not how it does it
- Use the word **MUST** (not "shall", which is ambiguous)

In [ ]:
# display the contents of the requirements yaml file.
with open("day2_data/data/requirements.yml", 'r') as stream:
    data_loaded = yaml.safe_load(stream)

print(yaml.dump(data_loaded, Dumper=MyDumper, default_flow_style=False))


# 3. Hazard Log

Open `data/risk.yml` and add a new hazard for HandOsteo based on the care process diagram.
An example hazard has been provided to show the structure.

### Rules

- **Hazard** — the potential source of harm (not the harm itself).
- **Causes** — what could lead to the hazardous situation.
- **Effects** — what happens as a result of the hazard.
- **Harm** — the impact on the patient or user.
- **Severity / Likelihood** — use the labels defined at the top of the file.
- **Risk control** — a mitigation that reduces the likelihood or severity to an acceptable level.
- Link your risk control back to a requirement using a `linked_risk_control` field in `requirements.yml`.

In [ ]:
# Display the contents of the risk yaml file
with open('day2_data/data/risk.yml', 'r') as stream:
    data_loaded = yaml.safe_load(stream)

print(yaml.dump(data_loaded, Dumper=MyDumper, default_flow_style=False))

# 4. Design Specification

Open `day2_data/data/requirements.yml` and add a design specification item (`sys_des_spec`) for the requirement you just wrote.

A good design spec item:
- Has **enough detail** to implement the feature correctly
- Does **not** prescribe specific function names or class structures (unless a particular framework is required, e.g. "use MONAI")
- Is written so that it can be **verified by an automated unit test**

In [ ]:
# Import the HandOsteo stub classes and testing tools
import sys
if '.' not in sys.path:
    sys.path.insert(0, '.')

from src.handosteo import DicomLoader, ViewClassifier, MCPMeasurer, ReportGenerator
import pydicom
import pytest
import ipytest

ipytest.autoconfig()

# Smoke-test: load one AP X-ray and run the full pipeline
loader  = DicomLoader()
clf     = ViewClassifier()
meas    = MCPMeasurer()
rep_gen = ReportGenerator()

ds     = loader.load('day2_data/xray/dicom/AP_1.dcm')
view   = clf.classify(ds)
result = meas.measure(ds)
report = rep_gen.generate(result)   # status defaults to 'success'

print(f"View           : {view}")
print(f"A              : {result['A']} mm")
print(f"B              : {result['B']} mm")
print(f"MCP            : {result['MCP']:.1f}%")
print(f"\nReport:\n{report.decode()}")

### HandOsteo pipeline demo

We have 5 generated Hand X-rays in `day2_data/xray/dicom/` — 3 AP views and 2 obliques. The APs are from patients with a range of bone health, while the obliques are from healthy patients.

We have also created a simple pipeline that mocks the full process of loading, classifying, measuring, and reporting on each image. The pipeline is not perfect — it will return a measurement for the obliques.

Each AP hand X-ray has a **reference diagnosis** stored in its DICOM metadata (from a DEXA scan taken at the same visit). The pipeline processes each image independently and returns its own classification — we can then compare the two.

The two oblique views should be **rejected** before they ever reach the measurement step.

In [ ]:
# Show sample hand X-rays: AP views (accepted) and oblique views (rejected)
from pathlib import Path

xray_dir = Path('day2_data/xray/dicom')
files = {
    'AP hand 1': xray_dir / 'AP_1.dcm',
    'AP hand 2': xray_dir / 'AP_2.dcm',
    'AP hand 3': xray_dir / 'AP_3.dcm',
    'Oblique 1\n(rejected view)': xray_dir / 'Ob_1.dcm',
    'Oblique 2\n(rejected view)': xray_dir / 'Ob_2.dcm',
}

fig, axes = plt.subplots(1, 5, figsize=(18, 6))
for ax, (title, path) in zip(axes, files.items()):
    ds = pydicom.dcmread(str(path))
    arr = ds.pixel_array
    accepted = 'Ob' not in path.name
    ax.imshow(arr, cmap='gray', aspect='equal')
    ax.set_title(title, fontsize=10,
                 color='#1a7f1a' if accepted else '#c0392b', fontweight='bold')
    ax.axis('off')
    border_color = '#1a7f1a' if accepted else '#c0392b'
    for spine in ax.spines.values():
        spine.set_edgecolor(border_color)
        spine.set_linewidth(3)
        spine.set_visible(True)

fig.suptitle(
    'HandOsteo input data — AP views (green ✓) are processed; oblique views (red ✗) are rejected',
    fontsize=11, y=1.01
)
plt.tight_layout()
plt.show()
print("AP views show the second metacarpal end-on, enabling reliable A and B measurement.")
print("Oblique views introduce foreshortening that makes cortical width measurement unreliable.")


In [ ]:
import sys
if '.' not in sys.path:
    sys.path.insert(0, '.')

from src.handosteo import DicomLoader, ViewClassifier, MCPMeasurer, ReportGenerator
from pathlib import Path
import pandas as pd

loader  = DicomLoader()
clf     = ViewClassifier()
meas    = MCPMeasurer()
rep_gen = ReportGenerator()

rows = []
for dcm_path in sorted(Path('day2_data/xray/dicom').glob('*.dcm')):
    ds = loader.load(str(dcm_path))
    reference = getattr(ds, 'PatientComments', '—').replace('Reference diagnosis: ', '')
    try:
        view    = clf.classify(ds)
        result  = meas.measure(ds)
        report  = rep_gen.generate(result)
        mcp     = result['MCP']
        outcome = report.decode().splitlines()[-1].replace('Classification: ', '')
        rows.append({
            'File':           dcm_path.name,
            'Reference (DEXA)': reference,
            'View':           view,
            'MCP (%)':        f'{mcp:.1f}',
            'HandOsteo says': outcome,
            'Status':         '✓ ACCEPTED',
        })
    except ValueError:
        rows.append({
            'File':           dcm_path.name,
            'Reference (DEXA)': reference,
            'View':           getattr(ds, 'ViewPosition', '?'),
            'MCP (%)':        '—',
            'HandOsteo says': '—',
            'Status':         '✗ REJECTED',
        })

df = pd.DataFrame(rows)
display(df.to_html(index=False), metadata={'text/html': True})
print(df.to_string(index=False))

# 5. Verification

Now it's your turn to write unit tests for the **HandOsteo** pipeline.

HandOsteo takes an AP hand X-ray (DICOM), classifies the view, measures the second metacarpal, and returns a report. We have provided a set of **stub classes** in `src/handosteo.py` that replicate the interface of the real system without needing a trained model.

Your goal is to write unit tests that **verify the design specification items** you wrote above (or you can use the ones in the solution, if that's easier. Just turn on LOAD_SOLUTIONS.)

### The pipeline classes

| Class | What it does |
|---|---|
| `DicomLoader` | Loads a DICOM file and validates its modality (CR or DX) |
| `ViewClassifier` | Classifies the view as AP, PA, or raises for oblique |
| `MCPMeasurer` | Measures A, B and calculates MCP = ((A−B)/A)×100 |
| `ReportGenerator` | Generates a byte-string report for success or failure |

### Test data

Real (de-identified) hand X-ray DICOMs are in `day2_data/xray/dicom/`:

| File | View |
|---|---|
| `AP_1.dcm`, `AP_2.dcm`, `AP_3.dcm` | AP hand X-rays (should be accepted) |
| `Ob_1.dcm`, `Ob_2.dcm` | Oblique views (should be rejected) |

### What to test

For each SDS item, write at least one test that would **fail** if the implementation were broken. Think about:

- **Happy path** — does valid input produce the correct output?
- **Edge cases** — what boundary values matter?
- **Error paths** — what invalid inputs must be rejected, and with what exception?

Use `pytest.raises(...)` to assert that the right exceptions are raised.

In [ ]:
# Import the HandOsteo stub classes and testing tools
import sys
if 'src' not in sys.path:
    sys.path.insert(0, '.')

from src.handosteo import DicomLoader, ViewClassifier, MCPMeasurer, ReportGenerator
import pydicom
import pytest
import ipytest

ipytest.autoconfig()

# Smoke-test: load one AP X-ray and run the full pipeline
loader  = DicomLoader()
clf     = ViewClassifier()
meas    = MCPMeasurer()
rep_gen = ReportGenerator()

ds     = loader.load('day2_data/xray/dicom/Ob_1.dcm')
view   = clf.classify(ds)
result = meas.measure(ds)
report = rep_gen.generate(result)

print(f"View : {view}")
print(f"A    : {result['A']} mm")
print(f"B    : {result['B']} mm")
print(f"MCP  : {result['MCP']:.1f}%")
print(f"\nReport preview:\n{report.decode()}")

> ### ✏️ Your task
>
> Write at least one test per SDS item in the class below.
> Use `pytest.raises(SomeException)` to test error paths.
> Run the cell to see which tests pass.
>
> **Hint:** you can create a minimal in-memory DICOM dataset for mocking like this:
> ```python
> import pydicom
> ds = pydicom.Dataset()
> ds.Modality = "MG"   # wrong modality — loader should reject this
> ```

In [ ]:
class TestHandOsteo:
    pass  # write your tests here


ipytest.run('-vv')

> ### 💡 Stuck? Load the example unit tests
>
> If you'd like to see a worked example, set `LOAD_SOLUTIONS = True` in the cell below and run it.
> The solution tests from `src/unit_tests.py` will be run directly so you can read through them.
>
> ⚠️ Try to write your own tests first — the goal is to practise thinking like a verification engineer.

In [ ]:
# 💡 Load the worked HandOsteo unit tests
# Set LOAD_SOLUTIONS = True and run this cell to see the example test suite.

LOAD_SOLUTIONS = False  # <-- change to True

if LOAD_SOLUTIONS:
    from IPython.display import display, Markdown
    import subprocess

    # Show the solution source so students can read through it
    with open('src/unit_tests.py') as f:
        src = f.read()
    display(Markdown(f'```python\n{src}\n```'))

    # Run the solution tests and print the results
    result = subprocess.run(
        [sys.executable, '-m', 'pytest', 'src/unit_tests.py', '-v'],
        capture_output=True, text=True,
    )
    print(result.stdout)
    if result.stderr:
        print(result.stderr)
else:
    print('LOAD_SOLUTIONS is False — nothing shown. Set it to True above to see the example tests.')

# 6. Validation

Add a manual validation test to `day2_data/data/manual_tests.yml` that partially satisfies one of your requirements.

> *"Validation is confirming you've built the **right** software"* — it ensures the system fulfils the user's actual needs, not just the technical specification.

### Rules for manual test cases

- Must be executable by an end user (not necessarily the developer)
- Steps must be clearly written with a defined expected outcome
- Should map back to a system requirement (`SRS_XXX`)

In [ ]:
# display the contents of the requirements yaml file.
with open("day2_data/data/manual_tests.yml", 'r') as stream:
    data_loaded = yaml.safe_load(stream)

print(yaml.dump(data_loaded, Dumper=MyDumper, default_flow_style=False))

In [ ]:
# Build the documentation using the command 'make'
! cd day2_data && make && cd ..

In [ ]:
# 📄 See your completed QMS rendered into official documents
from IPython.display import Markdown, display
doc = 'day2_data/release/system-requirements.md'
if os.path.exists(doc):
    display(Markdown(open(doc).read()))
else:
    print('Run the make cell above first to generate', doc)

In [ ]:
# 📄 See your completed QMS rendered into official documents
doc = 'day2_data/release/hazard-log.md'
if os.path.exists(doc):
    display(Markdown(open(doc).read()))
else:
    print('Run the make cell above first to generate', doc)

In [ ]:
# 📄 See your completed QMS rendered into official documents
doc = 'day2_data/release/system-design-specification.md'
if os.path.exists(doc):
    display(Markdown(open(doc).read()))
else:
    print('Run the make cell above first to generate', doc)